# Run the site scraper on Google Colab

This notebook installs Scrapy + Playwright and runs one of the spiders in this repo (`oecd`, `agendastad`, `elkeregiotelt`, `digitaleconomy`).

**Default off-site behaviour.** Each crawl stays recursively inside its root domain(s). For every external link the root site has, the spider does a single-page visit to that URL — just to save its text/markdown and download any PDFs referenced on it — and stops there. No approval is needed for this.

**Optional escalation.** If a particular external domain turns out to be worth a full recursive crawl, rerun the spider with that domain approved; see section 7.

**Keep this tab open** while a crawl runs — Colab disconnects idle sessions after ~90 min.

## 1. Install dependencies

In [ ]:
!pip install -q scrapy scrapy-playwright markdownify beautifulsoup4 lxml Pillow

In [ ]:
# Chromium + the Linux libs it needs. Colab runs the kernel as root so no sudo is needed.
!playwright install chromium
!playwright install-deps chromium

## 2. Get the code

In [ ]:
%%bash
set -euo pipefail
REPO_DIR=/content/google-scholar-scrapy-spider
BRANCH=claude/build-website-scraper-AOTSB
if [ ! -d "$REPO_DIR/.git" ]; then
  git clone --branch "$BRANCH" https://github.com/GdeJoode/google-scholar-scrapy-spider.git "$REPO_DIR"
else
  git -C "$REPO_DIR" fetch origin "$BRANCH"
  git -C "$REPO_DIR" checkout "$BRANCH"
  git -C "$REPO_DIR" pull --ff-only origin "$BRANCH"
fi

In [ ]:
%cd /content/google-scholar-scrapy-spider
%env SCRAPY_PROJECT=site

## 3. (Optional) Persist output to Google Drive

Colab's `/content/` storage is wiped at the end of the session. Mount Drive and symlink `output/` into it to keep results and the approved-off-site list between runs.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!mkdir -p /content/drive/MyDrive/scraper_output
!ln -sfn /content/drive/MyDrive/scraper_output output
!ls -la output

## 4. Reset cache + output (use when re-running from scratch)

Scrapy's HTTP cache stores every response, including 403 failures. Wipe the cache (and the output dir, unless it's a Drive symlink) when you change settings or want a fresh crawl.

In [ ]:
!rm -rf httpcache .scrapy
![ ! -L output ] && rm -rf output || echo 'output is a Drive symlink, not clearing'

## 5. Crawl a spider

The crawl covers the root site + one landing-page visit for every external URL it links to (text + PDFs). Start with `CLOSESPIDER_PAGECOUNT` for a quick smoke test; remove it for the real run.

In [ ]:
# Test run — stops after 25 pages so you can check the output shape.
!scrapy crawl elkeregiotelt -s CLOSESPIDER_PAGECOUNT=25

In [ ]:
!ls -la output/elkeregiotelt/
!wc -l output/elkeregiotelt/pages.jsonl output/elkeregiotelt/publications.jsonl output/elkeregiotelt/downloads_manifest.jsonl 2>/dev/null

In [ ]:
# Full phase-1 crawls. Run only the spider(s) you care about.

# !scrapy crawl agendastad
# !scrapy crawl elkeregiotelt
# !scrapy crawl oecd
# !scrapy crawl digitaleconomy

## 6. Review external domains

After phase 1, aggregate all recorded external URLs and write a human-editable candidates list. Change `elkeregiotelt` below to whichever spider you just ran.

The helper writes `output/<spider>/approved_off_site.txt` with every candidate **commented out** (`# domain`). Open that file, delete the `# ` from the lines you want to include, save, then continue to phase 2.

In [ ]:
!python -m site_scraper.review_externals elkeregiotelt --top 60

In [ ]:
# Open the approved-list in Colab's editor by clicking the file in the left sidebar,
# or just cat/edit it here. Uncomment the domains you want to crawl in phase 2.
!cat output/elkeregiotelt/approved_off_site.txt

## 7. (Optional) Escalate specific external domains to a full recursive crawl

Open `output/<spider>/approved_off_site.txt`, remove the `# ` from the domains you want to crawl recursively, and re-run the spider. Same-domain links inside those domains will now be followed like root-site links; links to yet another domain are still only recorded in `external_links`.

In [ ]:
!scrapy crawl elkeregiotelt \
    -a off_site_from_file=output/elkeregiotelt/approved_off_site.txt

In [ ]:
# Alternative: pass domains inline without editing the file.
# !scrapy crawl elkeregiotelt -a off_site_domains=platformisor.nl,citydeal-foo.nl

## 8. Download the output

Skip this cell if you mounted Drive — the files are already in `MyDrive/scraper_output`.

In [ ]:
import shutil
from google.colab import files

archive = shutil.make_archive('/content/scraper_output', 'zip', root_dir='output')
print('Wrote', archive)
files.download(archive)